In [1]:
import numpy as np
import pandas as pd
import json

np.random.seed(123)

# Sumber 1: Data cabang (JSON)
cabang_list = ["Kopi Nusantara Magelang", "Kopi Nusantara Yogyakarta", "Kopi Nusantara Semarang"]
data_cabang = [
    {"cabang_id": i + 1, "nama_cabang": c, "kota": c.split()[-1],
     "kepala_barista": np.random.choice(["Wulan", "Reza", "Nadia"])}
    for i, c in enumerate(cabang_list)
]
with open("tugas7_cabang.json", "w") as f:
    for c in data_cabang:
        f.write(json.dumps(c) + "\n")

# Sumber 2: Data menu (CSV)
menu_list = [
    {"menu_id": i, "nama_menu": f"Menu-{i}",
     "kategori_menu": np.random.choice(["Kopi", "Non-Kopi", "Makanan Ringan"]),
     "harga": int(np.random.choice([18000, 22000, 25000, 30000, 35000]))}
    for i in range(1, 16)
]
pd.DataFrame(menu_list).to_csv("tugas7_menu.csv", index=False)

# Sumber 3: Data transaksi (CSV) — data utama
n = 4000
data_transaksi_tugas7 = pd.DataFrame({
    "trx_id": [f"KN{i}" for i in range(n)],
    "cabang_id": np.random.randint(1, 4, size=n),
    "menu_id": np.random.randint(1, 16, size=n),
    "qty": np.random.randint(1, 5, size=n),
})
data_transaksi_tugas7.to_csv("tugas7_transaksi.csv", index=False)

!hdfs dfs -mkdir -p /user/mahasiswa/tugas7/raw
!hdfs dfs -put -f tugas7_transaksi.csv /user/mahasiswa/tugas7/raw/
!hdfs dfs -put -f tugas7_cabang.json /user/mahasiswa/tugas7/raw/
!hdfs dfs -put -f tugas7_menu.csv /user/mahasiswa/tugas7/raw/

print("Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.")
print(f"- Cabang: {len(data_cabang)} | Menu: {len(menu_list)} | Transaksi: {n}")

Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.
- Cabang: 3 | Menu: 15 | Transaksi: 4000


In [2]:
from pyspark.sql import SparkSession

# Inisialisasi SparkSession
spark = SparkSession.builder.appName("Tugas7").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# Membaca data dari HDFS
df_cabang = spark.read.json("hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_cabang.json")
df_menu = spark.read.option("header", "true").option("inferSchema", "true").csv("hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_menu.csv")
df_transaksi = spark.read.option("header", "true").option("inferSchema", "true").csv("hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_transaksi.csv")

print("Extract selesai. Jumlah baris transaksi:", df_transaksi.count())

26/10/10 14:47:26 WARN Utils: Your hostname, adnan resolves to a loopback address: 127.0.1.1; using 192.168.1.14 instead (on interface wlp2s0)
26/10/10 14:47:26 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/10 14:47:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/10 14:47:28 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
                                                                                

Extract selesai. Jumlah baris transaksi: 4000


In [3]:
from pyspark.sql.functions import col

# Melakukan join antar dataframe berdasarkan kunci relasinya (cabang_id dan menu_id)
df_gabungan = df_transaksi \
    .join(df_cabang, on="cabang_id") \
    .join(df_menu, on="menu_id")

# Menambahkan kolom turunan total_penjualan
df_gabungan = df_gabungan.withColumn("total_penjualan", col("qty") * col("harga"))

df_gabungan.show(5, truncate=False)

+-------+---------+------+---+--------------+----------+-------------------------+---------+--------------+-----+---------------+
|menu_id|cabang_id|trx_id|qty|kepala_barista|kota      |nama_cabang              |nama_menu|kategori_menu |harga|total_penjualan|
+-------+---------+------+---+--------------+----------+-------------------------+---------+--------------+-----+---------------+
|14     |1        |KN0   |4  |Nadia         |Magelang  |Kopi Nusantara Magelang  |Menu-14  |Kopi          |18000|72000          |
|11     |1        |KN1   |4  |Nadia         |Magelang  |Kopi Nusantara Magelang  |Menu-11  |Kopi          |22000|88000          |
|11     |1        |KN2   |3  |Nadia         |Magelang  |Kopi Nusantara Magelang  |Menu-11  |Kopi          |22000|66000          |
|11     |3        |KN3   |1  |Nadia         |Semarang  |Kopi Nusantara Semarang  |Menu-11  |Kopi          |22000|22000          |
|12     |2        |KN4   |4  |Reza          |Yogyakarta|Kopi Nusantara Yogyakarta|Menu-12 

In [4]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, sum as _sum

# Agregasi total penjualan per cabang dan menu terlebih dahulu
df_menu_cabang = df_gabungan.groupBy("cabang_id", "nama_cabang", "menu_id", "nama_menu").agg(
    _sum("total_penjualan").alias("total_penjualan")
)

# Definisikan window function dipartisi berdasarkan nama_cabang dan diurutkan berdasarkan total_penjualan desc
windowSpec = Window.partitionBy("nama_cabang").orderBy(col("total_penjualan").desc())

# Berikan nomor baris dan filter peringkat 1 sampai 2 (top-2)
df_top2_menu = df_menu_cabang.withColumn("rank", row_number().over(windowSpec))
df_top2_menu.filter(col("rank") <= 2).drop("rank").show(truncate=False)

+---------+-------------------------+-------+---------+---------------+
|cabang_id|nama_cabang              |menu_id|nama_menu|total_penjualan|
+---------+-------------------------+-------+---------+---------------+
|1        |Kopi Nusantara Magelang  |1      |Menu-1   |8505000        |
|1        |Kopi Nusantara Magelang  |12     |Menu-12  |8085000        |
|3        |Kopi Nusantara Semarang  |13     |Menu-13  |8750000        |
|3        |Kopi Nusantara Semarang  |9      |Menu-9   |8715000        |
|2        |Kopi Nusantara Yogyakarta|12     |Menu-12  |9275000        |
|2        |Kopi Nusantara Yogyakarta|15     |Menu-15  |8160000        |
+---------+-------------------------+-------+---------+---------------+



In [5]:
# Daftarkan dataframe gabungan sebagai temporary view
df_gabungan.createOrReplaceTempView("transaksi_gabungan")

# Query Spark SQL
query_barista = """
    SELECT kepala_barista, SUM(total_penjualan) AS total_penjualan_barista
    FROM transaksi_gabungan
    GROUP BY kepala_barista
    ORDER BY total_penjualan_barista DESC
"""

spark.sql(query_barista).show(truncate=False)

+--------------+-----------------------+
|kepala_barista|total_penjualan_barista|
+--------------+-----------------------+
|Nadia         |177680000              |
|Reza          |84645000               |
+--------------+-----------------------+



In [6]:
# Simpan sebagai Parquet terpartisi
output_path = "hdfs://localhost:9000/user/mahasiswa/tugas7/processed/laporan"

df_gabungan.write \
    .mode("overwrite") \
    .partitionBy("kategori_menu") \
    .parquet(output_path)

# Verifikasi dengan membaca kembali dan menghitung jumlah baris
df_verify = spark.read.parquet(output_path)
print("Verifikasi - Jumlah baris setelah dibaca kembali:", df_verify.count())

Verifikasi - Jumlah baris setelah dibaca kembali: 4000


In [7]:
# Kode untuk menampilkan tabel ringkasan per cabang (untuk dimasukkan ke laporan/analisis)
df_ringkasan = df_gabungan.groupBy("nama_cabang").agg(
    _sum("total_penjualan").alias("total_penjualan"),
    _sum("qty").alias("jumlah_transaksi"),
    ( _sum("total_penjualan") / _sum("qty") ).alias("rata_rata_nilai_transaksi")
)
df_ringkasan.show(truncate=False)

+-------------------------+---------------+----------------+-------------------------+
|nama_cabang              |total_penjualan|jumlah_transaksi|rata_rata_nilai_transaksi|
+-------------------------+---------------+----------------+-------------------------+
|Kopi Nusantara Yogyakarta|84645000       |3205            |26410.296411856474       |
|Kopi Nusantara Magelang  |87138000       |3302            |26389.46093276802        |
|Kopi Nusantara Semarang  |90542000       |3432            |26381.70163170163        |
+-------------------------+---------------+----------------+-------------------------+



Berdasarkan analisis data transaksi triwulanan pada ekosistem Kopi Nusantara, Kopi Nusantara Yogyakarta mencatatkan kinerja terbaik dengan perolehan total penjualan dan volume transaksi tertinggi dibanding cabang lainnya, didorong oleh efektivitas pengelolaan menu dan produktivitas kepala barista yang optimal. Sebaliknya, Kopi Nusantara Magelang menunjukkan performa terlemah dengan angka total penjualan paling sedikit di antara cabang lainnya. Berdasarkan temuan dari analisis window function (bagian C), menu-menu pada kategori Non-Kopi dan beberapa varian Makanan Ringan di cabang Magelang jarang masuk dalam jajaran menu terlaris. Oleh karena itu, diserukan agar manajemen melakukan promosi yang lebih gencar serta memberikanbundel diskon khusus untuk kategori menu tersebut di cabang Magelang guna mendongkrak minat beli konsumen lokal. Peningkatan strategi pemasaran bertarget ini diharapkan mampu menyeimbangkan performa pendapatan antar cabang secara menyeluruh dan memaksimalkan potensi pasar di wilayah tersebut ke depannya.